## K-means Clustering

This notebook accompanies the **Clustering** slides that we are discussing in class. We cluster 342 Palmer penguins using four body measurements, **without** telling the algorithms the species. Species is used only to check how well the clusters line up with real structure. In a real clustering problem we would have no ability to check this.

*AI disclosure: The code in this notebook was generated with the assistance of Claude. I gave Claude a pretty detailed prompt, then I went through and modified both the code itself and the explanations provided. I made sure I could execute and validate the output for code written.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.datasets import make_moons
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform

# Same colors as the slides: orange, blue, teal, purple
PALETTE = ["#E57100", "#3E6FB0", "#2A9D8F", "#8E4FA0"]
sns.set_theme(style="white", palette=PALETTE)

### Load the penguins data

We are going to cluster the penguins using the numeric variables in the dataset: bill length, bill depth, flipper length, body mass. 

In [ ]:
penguins = pd.read_csv("../data/penguins.csv")
feats = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

# Keep the penguins with all four measurements
df = penguins.dropna(subset=feats).reset_index(drop=True)
X = df[feats]
print(df.shape)
df["species"].value_counts()

In [ ]:
# As usual, let's look at and understand the scale of our data prior to starting. 
X.describe().T[["mean", "std", "min", "max"]].round(1)

Body mass is in the thousands (grams) while bill depth is under 25 (mm). K-means and hierarchical clustering both use distances, so without scaling, body mass would dominate. We will use the `StandardScaler()`, as per usual. 

In [ ]:
# Here we transform all of the data immediately. We can't use a holdout set in the same way as supervised learning. 
scaler = StandardScaler()
Xs = scaler.fit_transform(X)   # a NumPy array: 342 rows x 4 standardized features
pd.DataFrame({"mean": Xs.mean(axis=0), "sd": Xs.std(axis=0)}, index=feats).round(3) 

In [ ]:
from itertools import combinations

labels = {"bill_length_mm": "Bill length (mm)", "bill_depth_mm": "Bill depth (mm)",
          "flipper_length_mm": "Flipper length (mm)", "body_mass_g": "Body mass (g)"}
pairs = list(combinations(feats, 2))   # all 6 pairs of the 4 features

fig, axes = plt.subplots(6, 2, figsize=(12, 27))
for k, (xf, yf) in enumerate(pairs):
    ax_obs, ax_lab = axes[k]
    ax_obs.scatter(df[xf], df[yf], color="gray", s=10, alpha=0.7)
    ax_obs.set_title("What we observe")
    sns.scatterplot(data=df, x=xf, y=yf, hue="species", style="species", s=15,
                    ax=ax_lab, legend=(k == 0))   # one legend is enough
    ax_lab.set_title("What we never see: the labels")
    for ax in (ax_obs, ax_lab):
        ax.set_xlabel(labels[xf])
        ax.set_ylabel(labels[yf])
plt.tight_layout()
plt.show()


## K-means

K-means looks for the partition $C_1, \ldots, C_K$ that minimizes the total **within-cluster sum of squares (WCSS)**:

$$\underset{C_1,\ldots,C_K}{\text{minimize}} \; \sum_{k=1}^{K}\sum_{i \in C_k} \lVert x_i - \mu_k \rVert^2$$

where $\mu_k$ is the centroid (mean) of cluster $k$. Trying every partition is impossible (about $10^{162}$ ways to split 342 penguins into 3 groups), so we use a greedy algorithm.

### Lloyd's algorithm by hand

ISL Algorithm 12.2:
1. **Initialize:** randomly assign each observation to one of the $K$ clusters.
2. **Update:** compute each cluster's centroid.
3. **Assign:** move each observation to the cluster with the nearest centroid.
4. **Repeat** steps 2 and 3 until the assignments stop changing.

In [ ]:
def wcss(Z, labels, centroids):
    # Total within-cluster sum of squares
    return sum(((Z[labels == k] - centroids[k]) ** 2).sum() for k in range(len(centroids)))


def lloyd(Z, K, seed=0, max_iter=100):
    # K-means as in ISL Algorithm 12.2. Returns final labels, centroids, and the history of each iteration.
    # (A real implementation also has to handle a cluster that ends up empty; we skip that here.)
    rng = np.random.default_rng(seed)
    labels = rng.integers(0, K, size=len(Z))                                  # 1. random assignment
    history = []
    for it in range(1, max_iter + 1):
        centroids = np.array([Z[labels == k].mean(axis=0) for k in range(K)])  # 2. update
        sq_dist = ((Z[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)    # n x K squared distances
        new_labels = sq_dist.argmin(axis=1)                                    # 3. assign
        history.append({"iteration": it, "labels": new_labels, "centroids": centroids,
                        "wcss": wcss(Z, new_labels, centroids)})
        if np.array_equal(new_labels, labels):                                 # 4. stop when nothing changes
            break
        labels = new_labels
    return labels, centroids, history

We use just two standardized features (bill length and depth) so we can watch the algorithm work.

In [ ]:
Z2 = StandardScaler().fit_transform(df[["bill_length_mm", "bill_depth_mm"]])
labels2, cent2, hist = lloyd(Z2, K=3, seed=7)

print(f"Converged after {len(hist)} iterations")
pd.DataFrame({"iteration": [h["iteration"] for h in hist],
              "WCSS": [round(h["wcss"], 1) for h in hist]}).set_index("iteration").T

In [ ]:
show = [0, 1, 2, len(hist) - 1]
fig, axes = plt.subplots(1, 4, figsize=(17, 4.2), sharex=True, sharey=True)
for ax, i in zip(axes, show):
    h = hist[i]
    ax.scatter(Z2[:, 0], Z2[:, 1], c=[PALETTE[k] for k in h["labels"]], s=12, alpha=0.7)
    ax.scatter(h["centroids"][:, 0], h["centroids"][:, 1], c=PALETTE[:3], marker="X", s=250, edgecolor="black")
    ax.set_title(f"Iteration {h['iteration']}: WCSS = {h['wcss']:.1f}")
    ax.set_xlabel("Bill length (std.)")
axes[0].set_ylabel("Bill depth (std.)")
plt.tight_layout()
plt.show()

In iteration 1 the three centroids sit almost on top of each other near the middle: a random assignment gives three clusters with nearly the same mean. They spread out quickly after that.

**WCSS never goes up.** The update step can only lower WCSS (the mean minimizes squared distances), and so can the assign step (each point moves to a closer centroid). With finitely many partitions, the algorithm has to stop, but only at a **local** optimum.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot([h["iteration"] for h in hist], [h["wcss"] for h in hist], marker="o", color="#232D4B")
ax.set_xlabel("Iteration")
ax.set_ylabel("WCSS")
plt.show()

### K-means with scikit-learn

Now all four standardized features. Note that `fit` takes only `X`: there is no `y` in clustering.

In [ ]:
km = KMeans(n_clusters=3, n_init=20, random_state=0)
km.fit(Xs)

print(f"WCSS (inertia_): {km.inertia_:.1f}")
print("Cluster sizes:", np.bincount(km.labels_))
km.cluster_centers_.round(2)   # centroids, in standardized units

### Local optima and restarts

Run K-means 30 times with **one** random start each (`init="random"`, `n_init=1`) and look at the distribution of the final WCSS scores for these different seeds.

In [ ]:
runs = [KMeans(n_clusters=3, init="random", n_init=1, random_state=s).fit(Xs).inertia_ for s in range(30)]
pd.Series(np.round(runs, 1)).value_counts().sort_index().rename_axis("final WCSS").rename("number of runs")

Some runs land at WCSS = 379.4; others get stuck near 486, a worse local optimum.

Sklearn has two ways of handling this for us:

* **Restarts:** run K-means `n_init` times and keep the run with the lowest WCSS.
* **K-means++:** choose starting centers that are spread out. Each new center is picked with probability proportional to $D(x_i)^2$, the squared distance to the nearest center already chosen. This is sklearn's default `init`.

With the default `n_init="auto"`, sklearn runs K-means++ only **once**. 

In [ ]:
models = {
    "random init, 1 run (seed 3)": KMeans(3, init="random", n_init=1, random_state=3),
    "k-means++, n_init='auto' = 1 run (seed 2)": KMeans(3, random_state=2),
    "random init, n_init=20 (seed 2)": KMeans(3, init="random", n_init=20, random_state=2),
    "k-means++, n_init=20 (seed 2)": KMeans(3, n_init=20, random_state=2),
}
pd.Series({name: round(m.fit(Xs).inertia_, 1) for name, m in models.items()}, name="WCSS")

### Interpreting the clusters

Describe each cluster by its feature means **in the original units**. That is how clusters get "names". (Cluster numbers are arbitrary; they can change between runs.)

In [ ]:
km.labels_

In [ ]:
# Here, we are adding an additional column to our dataframe with the assigned cluster for each penguin
# Then we are looking at the mean values for each. 
df["kmeans"] = km.labels_
profile = df.groupby("kmeans")[feats].mean().round(1)
profile.insert(0, "n", df["kmeans"].value_counts().sort_index())
profile

The centroids from `km.cluster_centers_` give the same numbers if we scale back to the original units

In [ ]:
# Unscaled
pd.DataFrame(km.cluster_centers_, columns=feats).round(1)

In [ ]:
# Scaled
pd.DataFrame(scaler.inverse_transform(km.cluster_centers_), columns=feats).round(1)

**Do the clusters match species?** We can check here only because we happen to know the species.

In [ ]:
pd.crosstab(df["kmeans"], df["species"])

One cluster is exactly the 123 Gentoo penguins. Adelie and Chinstrap overlap more: 24 Adelie are grouped with most of the Chinstraps.

## Choosing K

There is no `y`, so there is no validation error to minimize, and WCSS **always** decreases as K grows (with K = n every point is its own cluster and WCSS = 0). We need other tools.

### 6. The elbow method

In [ ]:
Ks = range(1, 11)
rows = []
for k in Ks:
    # Remember, this is doing KMeans++ by default
    m = KMeans(n_clusters=k, n_init=20, random_state=0).fit(Xs)
    rows.append({"K": k, "WCSS": m.inertia_,
                 "avg silhouette": silhouette_score(Xs, m.labels_) if k > 1 else np.nan})
choose_k = pd.DataFrame(rows).set_index("K")
choose_k.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(choose_k.index, choose_k["WCSS"], marker="o", color="#232D4B")
ax.set_xlabel("Number of clusters K")
ax.set_ylabel("Total WCSS (inertia_)")
ax.set_xticks(list(Ks))
plt.show()

### Silhouette scores

For each observation $i$:
* $a(i)$ = average distance from $i$ to the other points in **its own** cluster
* $b(i)$ = average distance from $i$ to the points in the **nearest other** cluster

$$s(i) = \frac{b(i) - a(i)}{\max\{a(i),\, b(i)\}}$$

$s(i)$ near 1 means well clustered, near 0 means on a boundary, and negative means $i$ is probably in the wrong cluster. Averaging $s(i)$ over all points scores a whole clustering.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(choose_k.index[1:], choose_k["avg silhouette"].iloc[1:], marker="o", color="#232D4B")
ax.set_xlabel("Number of clusters K")
ax.set_ylabel("Average silhouette")
ax.set_xticks(list(Ks)[1:])
plt.show()

best_k = choose_k["avg silhouette"].idxmax()
print(f"Largest average silhouette: K = {best_k} ({choose_k.loc[best_k, 'avg silhouette']:.3f})")

The silhouette prefers **K = 2**: Gentoo vs. everyone else. Look at the individual $s(i)$ values for K = 3:

In [ ]:
s_i = silhouette_samples(Xs, km.labels_)
pd.DataFrame({"cluster": km.labels_, "s": s_i}).groupby("cluster")["s"].agg(
    n="size", mean="mean", negative=lambda s: (s < 0).sum()).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
y0 = 0
for k in range(3):
    vals = np.sort(s_i[km.labels_ == k])
    ax.fill_betweenx(np.arange(y0, y0 + len(vals)), 0, vals, color=PALETTE[k])
    ax.text(-0.05, y0 + len(vals) / 2, f"Cluster {k}", ha="right", va="center")
    y0 += len(vals) + 10
ax.axvline(s_i.mean(), color="#232D4B", ls="--", label=f"average = {s_i.mean():.2f}")
ax.axvline(0, color="gray", lw=0.8)
ax.set_yticks([])
ax.set_xlabel("Silhouette s(i)")
ax.legend(loc="lower right")
plt.show()

### The gap statistic

Compare $\log(W_K)$ (the log WCSS) with what we would expect for data with **no** cluster structure:

$$\text{Gap}(K) = E^{*}[\log(W_K)] - \log(W_K)$$

$E^*$ is an average over $B$ reference data sets drawn uniformly over the range of each feature. Rule of thumb (Tibshirani, Walther & Hastie, 2001): choose the smallest $K$ such that

$$\text{Gap}(K) \geq \text{Gap}(K+1) - s_{K+1}$$

where $s_K$ is the simulation standard error. sklearn doesn't include the gap statistic, so we write it ourselves.

In [ ]:
def gap_statistic(Z, K_values, B=50, seed=0):
    rng = np.random.default_rng(seed)

    #Find the smallest and largest value of each feature
    lo, hi = Z.min(axis=0), Z.max(axis=0) 

    # Create B fake datasets that are the same size as Z. 
    # Every point is drawn uniformly at random so there should be no cluster structure
    refs = [rng.uniform(lo, hi, size=Z.shape) for _ in range(B)]

    # Fit Kmeans for each set of data and value of K and return the log of WCSS
    def log_wk(data, k):
        return np.log(KMeans(n_clusters=k, n_init=10, random_state=0).fit(data).inertia_)

    rows = []
    for k in K_values:
        # Compute the log for each of the fake datasets
        ref_logs = np.array([log_wk(R, k) for R in refs])

        # Find the difference in the mean of the log of the fake data and our actual WCSS
        # Calculate the standard error, since we want the smallest K with gap > gap(K+1) - s
        rows.append({"K": k, "gap": ref_logs.mean() - log_wk(Z, k),
                     "s": ref_logs.std() * np.sqrt(1 + 1 / B)})
    return pd.DataFrame(rows).set_index("K")


gap = gap_statistic(Xs, range(1, 9))   # takes a few seconds
gap["Gap(K+1) - s(K+1)"] = (gap["gap"] - gap["s"]).shift(-1)
gap["rule met?"] = gap["gap"] >= gap["Gap(K+1) - s(K+1)"]
k_gap = gap.index[gap["rule met?"]][0]
print(f"Gap statistic picks K = {k_gap}")
gap.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.errorbar(gap.index, gap["gap"], yerr=gap["s"], marker="o", capsize=4, color="#232D4B")
ax.set_xlabel("Number of clusters K")
ax.set_ylabel("Gap(K)")
plt.show()

### 9. The methods disagree

| Method | Suggested K |
|---|---|
| Elbow | 3 (maybe 4) |
| Average silhouette | 2 |
| Gap statistic | 5 |
| Species (known here) | 3 |

This is normal. Treat these as evidence, not answers: try a few values of K, compare the cluster profiles, and choose one that is stable and useful for the question at hand.

### 10. When K-means struggles

K-means assigns each point to the **nearest centroid**, so it favors round clusters of similar size. On two interlocking half-moons (simulated data) it cuts straight across them:

In [ ]:
Xm, _ = make_moons(n_samples=300, noise=0.06, random_state=1)
moon_km = KMeans(n_clusters=2, n_init=20, random_state=0).fit(Xm)

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(Xm[:, 0], Xm[:, 1], c=[PALETTE[k] for k in moon_km.labels_], s=15)
ax.scatter(*moon_km.cluster_centers_.T, marker="X", s=250, c=PALETTE[:2], edgecolor="black")
ax.set_title("K-means on two moons (simulated)")
plt.show()

We'll come back to the moons after we see hierarchical clustering.